# 02 — Prepare the pilot dataset

This clean notebook extracts a controlled pilot corpus from the ETCBC/BHSA Hebrew Bible corpus and exports it as analysis-ready flat tables.

It intentionally keeps only one authoritative chapter selection, using BHSA book names exactly as Text-Fabric reports them.

## 1. Setup and load BHSA

The project uses Text-Fabric to access the ETCBC/BHSA corpus. Raw BHSA data is not stored in this repository; the notebook loads it directly through Text-Fabric.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/ALPcourse_Biblical_Hebrew_Project/biblical_hebrew_genre_analysis")
NOTEBOOK_DIR = PROJECT_DIR / "notebooks"
DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = PROJECT_DIR / "output"
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"
DOCS_DIR = PROJECT_DIR / "docs"
POSTER_DIR = PROJECT_DIR / "poster"

for folder in [PROCESSED_DIR, TABLES_DIR, FIGURES_DIR, DOCS_DIR, POSTER_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

os.chdir(NOTEBOOK_DIR)
print("Project directory:", PROJECT_DIR)
print("Current folder:", os.getcwd())


!pip -q install text-fabric pandas

from tf.app import use
import pandas as pd

A = use('ETCBC/bhsa', hoist=globals())
print('BHSA loaded')
print('Number of word nodes:', len(F.otype.s('word')))

Mounted at /content/drive
Project directory: /content/drive/MyDrive/ALPcourse_Biblical_Hebrew_Project/biblical_hebrew_genre_analysis
Current folder: /content/drive/MyDrive/ALPcourse_Biblical_Hebrew_Project/biblical_hebrew_genre_analysis/notebooks
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 89.6 MB/s eta 0:00:00
**Locating corpus resources ...**
The requested app is not available offline
	~/text-fabric-data/github/ETCBC/bhsa/app not found
Status: latest release online v1.8.1 versus None locally
downloading app, main data and requested additions ...
app: ~/text-fabric-data/github/ETCBC/bhsa/app
data: ~/text-fabric-data/github/ETCBC/bhsa/tf/2021
   |     0.57s T otype                from ~/text-fabric-data/github/ETCBC/bhsa/tf/2021
   |       13s T oslots               from ~/text-fabric-data/github/ETCBC/bhsa/tf/2021
   |     0.00s T book@la              from ~/text-fabric-data/github/ETCBC/bhsa/tf/20

## 2. Inspect available book names

This prevents a common reproducibility problem: some English book names differ from the BHSA names. For example, BHSA uses `Deuteronomium`, `Jesaia`, `Psalmi`, and `Proverbia`.

In [2]:
books = [F.book.v(book_node) for book_node in F.otype.s("book")]
print("Number of books:", len(books))
print(books)

Number of books: 39
['Genesis', 'Exodus', 'Leviticus', 'Numeri', 'Deuteronomium', 'Josua', 'Judices', 'Samuel_I', 'Samuel_II', 'Reges_I', 'Reges_II', 'Jesaia', 'Jeremia', 'Ezechiel', 'Hosea', 'Joel', 'Amos', 'Obadia', 'Jona', 'Micha', 'Nahum', 'Habakuk', 'Zephania', 'Haggai', 'Sacharia', 'Maleachi', 'Psalmi', 'Iob', 'Proverbia', 'Ruth', 'Canticum', 'Ecclesiastes', 'Threni', 'Esther', 'Daniel', 'Esra', 'Nehemia', 'Chronica_I', 'Chronica_II']


## 3. Define the pilot corpus

The unit of selection is biblical chapters. The pilot corpus compares four broad genre groups:

- Narrative
- Law
- Prophecy
- Poetry/Wisdom

The sample is controlled and balanced enough for a pilot study, while still being small enough to inspect manually.

In [3]:
PILOT_CHAPTERS = {
    "Genesis": {"genre": "Narrative", "ranges": [(37, 45)], "notes": "Joseph narrative"},
    "Samuel_I": {"genre": "Narrative", "ranges": [(8, 12)], "notes": "Rise of monarchy / Samuel and Saul"},

    "Leviticus": {"genre": "Law", "ranges": [(1, 7)], "notes": "Sacrificial law"},
    "Deuteronomium": {"genre": "Law", "ranges": [(12, 16)], "notes": "Legal and cultic material"},

    "Jesaia": {"genre": "Prophecy", "ranges": [(1, 12)], "notes": "Opening prophetic oracles"},
    "Amos": {"genre": "Prophecy", "ranges": [(1, 9)], "notes": "Prophetic judgment speeches"},

    "Psalmi": {"genre": "Poetry_Wisdom", "ranges": [(1, 25)], "notes": "Poetic psalms"},
    "Proverbia": {"genre": "Poetry_Wisdom", "ranges": [(1, 10)], "notes": "Wisdom instruction"},
}

missing = sorted(set(PILOT_CHAPTERS) - set(books))
if missing:
    raise ValueError(f"These requested BHSA book names were not found: {missing}")

def in_ranges(chapter, ranges):
    return any(start <= int(chapter) <= end for start, end in ranges)

sample_rows = []
for book, info in PILOT_CHAPTERS.items():
    for start, end in info["ranges"]:
        sample_rows.append({
            "genre": info["genre"],
            "book": book,
            "start_chapter": start,
            "end_chapter": end,
            "notes": info["notes"],
        })

sample_df = pd.DataFrame(sample_rows)
sample_df.to_csv(PROCESSED_DIR / "genre_chapter_sample.csv", index=False)

genre_mapping = sample_df[["book", "genre"]].drop_duplicates()
genre_mapping.to_csv(PROCESSED_DIR / "genre_mapping.csv", index=False)

sample_df

,genre,book,start_chapter,end_chapter,notes
0,Narrative,Genesis,37,45,Joseph narrative
1,Narrative,Samuel_I,8,12,Rise of monarchy / Samuel and Saul
2,Law,Leviticus,1,7,Sacrificial law
3,Law,Deuteronomium,12,16,Legal and cultic material
4,Prophecy,Jesaia,1,12,Opening prophetic oracles
5,Prophecy,Amos,1,9,Prophetic judgment speeches
6,Poetry_Wisdom,Psalmi,1,25,Poetic psalms
7,Poetry_Wisdom,Proverbia,1,10,Wisdom instruction


## 4. Extract a word-level flat table

Each row represents one BHSA word token. The table includes textual location, genre, surface form, lexeme, part of speech, and optional Hebrew/gloss fields when available.

In [4]:
def safe_feature(feature_name, node):
    # Return a Text-Fabric feature value if it exists, otherwise None.
    try:
        return getattr(F, feature_name).v(node)
    except Exception:
        return None

rows = []

for book_node in F.otype.s("book"):
    book = F.book.v(book_node)
    if book not in PILOT_CHAPTERS:
        continue

    genre = PILOT_CHAPTERS[book]["genre"]
    ranges = PILOT_CHAPTERS[book]["ranges"]

    for chapter_node in L.d(book_node, otype="chapter"):
        chapter = int(F.chapter.v(chapter_node))
        if not in_ranges(chapter, ranges):
            continue

        for verse_node in L.d(chapter_node, otype="verse"):
            verse = int(F.verse.v(verse_node))

            for w in L.d(verse_node, otype="word"):
                rows.append({
                    "book": book,
                    "chapter": chapter,
                    "verse": verse,
                    "genre": genre,
                    "word_node": int(w),
                    "word": T.text(w).strip(),
                    "lex": F.lex.v(w),
                    "pos": F.sp.v(w),
                    "word_utf8": safe_feature("g_word_utf8", w),
                    "gloss": safe_feature("gloss", w),
                })

tokens = pd.DataFrame(rows)

if tokens.empty:
    raise ValueError("No tokens were extracted. Check book names and chapter ranges.")

tokens["chapter"] = tokens["chapter"].astype(int)
tokens["verse"] = tokens["verse"].astype(int)
tokens["word_node"] = tokens["word_node"].astype(int)

tokens.to_csv(PROCESSED_DIR / "biblical_hebrew_tokens.csv", index=False)
tokens.to_json(
    PROCESSED_DIR / "biblical_hebrew_tokens.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)

print("Total tokens:", len(tokens))
print("\nTokens by genre:")
print(tokens.groupby("genre").size().sort_values(ascending=False))
print("\nTokens by book:")
print(tokens.groupby(["genre", "book"]).size())

tokens.head()

Total tokens: 28215

Tokens by genre:
genre
Narrative        8527
Prophecy         7205
Poetry_Wisdom    6324
Law              6159
dtype: int64

Tokens by book:
genre          book         
Law            Deuteronomium    2633
               Leviticus        3526
Narrative      Genesis          5829
               Samuel_I         2698
Poetry_Wisdom  Proverbia        2698
               Psalmi           3626
Prophecy       Amos             2780
               Jesaia           4425
dtype: int64


,book,chapter,verse,genre,word_node,word,lex,pos,word_utf8,gloss
0,Genesis,37,1,Narrative,20165,וַ,W,conj,וַ,and
1,Genesis,37,1,Narrative,20166,יֵּ֣שֶׁב,JCB[,verb,יֵּ֣שֶׁב,sit
2,Genesis,37,1,Narrative,20167,יַעֲקֹ֔ב,J<QB/,nmpr,יַעֲקֹ֔ב,Jacob
3,Genesis,37,1,Narrative,20168,בְּ,B,prep,בְּ,in
4,Genesis,37,1,Narrative,20169,אֶ֖רֶץ,>RY/,subs,אֶ֖רֶץ,earth


## 5. Quality checks

These checks document the corpus size and make accidental changes visible. Exact counts can differ slightly across BHSA/Text-Fabric versions, so warnings are used instead of hard failures.

In [5]:
EXPECTED_TOTAL_TOKENS = 28215

if len(tokens) != EXPECTED_TOTAL_TOKENS:
    print(f"Warning: expected {EXPECTED_TOTAL_TOKENS} tokens, but extracted {len(tokens)}.")
    print("If Text-Fabric/BHSA versions changed, document the difference in the README.")
else:
    print("Token count matches the documented pilot corpus size.")

required_columns = ["book", "chapter", "verse", "genre", "word_node", "word", "lex", "pos"]
missing_columns = [col for col in required_columns if col not in tokens.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("\nExported files:")
print(PROCESSED_DIR / "biblical_hebrew_tokens.csv")
print(PROCESSED_DIR / "biblical_hebrew_tokens.jsonl")
print(PROCESSED_DIR / "genre_chapter_sample.csv")
print(PROCESSED_DIR / "genre_mapping.csv")

Token count matches the documented pilot corpus size.

Exported files:
/content/drive/MyDrive/ALPcourse_Biblical_Hebrew_Project/biblical_hebrew_genre_analysis/data/processed/biblical_hebrew_tokens.csv
/content/drive/MyDrive/ALPcourse_Biblical_Hebrew_Project/biblical_hebrew_genre_analysis/data/processed/biblical_hebrew_tokens.jsonl
/content/drive/MyDrive/ALPcourse_Biblical_Hebrew_Project/biblical_hebrew_genre_analysis/data/processed/genre_chapter_sample.csv
/content/drive/MyDrive/ALPcourse_Biblical_Hebrew_Project/biblical_hebrew_genre_analysis/data/processed/genre_mapping.csv


## 6. Data model

- One row = one word token.
- `genre` is assigned from the selected chapter range.
- `lex` is the BHSA lexeme/lemma code.
- `pos` is the BHSA part-of-speech category.
- `word_utf8` and `gloss` are included when available, but the main analysis uses `lex` for reproducibility.

The next notebook filters this table to content words and performs frequency and TF-IDF analysis.